In [ ]:
#Instalar
!pip install langchain-core
!pip install langchain-groq

In [ ]:
# =============================================================================
# ASSISTENTE ESPECIALISTA COM LANGCHAIN + GROQ
# =============================================================================
# O que este programa faz:
# - Pede o nome do usuário
# - Acessa um site e usa o conteúdo como base de conhecimento
# - Usa o modelo llama-3.3-70b-versatile via Groq
# - Responde perguntas com base no conteúdo do site
# - Interage chamando o usuário pelo nome
# - Permite várias perguntas
# - Encerra quando o usuário digitar "sair"
# =============================================================================


# -----------------------------------------------------------------------------
# PASSO 1 — IMPORTAR AS BIBLIOTECAS
# -----------------------------------------------------------------------------

# Biblioteca padrão do Python para variáveis de ambiente
import os

# Biblioteca para estruturar mensagens no LangChain
from langchain_core.messages import HumanMessage, SystemMessage

# Biblioteca para usar o Groq com LangChain
from langchain_groq import ChatGroq

# Biblioteca padrão para acessar páginas web
import urllib.request

# Biblioteca padrão para extrair texto do HTML
import html.parser


# -----------------------------------------------------------------------------
# PASSO 2 — CONFIGURAR API KEY DO GROQ
# -----------------------------------------------------------------------------

# Coloque sua API Key aqui
os.environ["GROQ_API_KEY"] = "Coloque Sua chave API"


# -----------------------------------------------------------------------------
# PASSO 3 — DEFINIR O SITE BASE DE CONHECIMENTO
# -----------------------------------------------------------------------------

# Site que será usado como fonte de conhecimento
URL_DO_SITE = "https://en.wikipedia.org/wiki/Pragmata"


# -----------------------------------------------------------------------------
# PASSO 4 — FUNÇÃO PARA EXTRAIR TEXTO DO SITE
# -----------------------------------------------------------------------------

def extrair_texto_do_site(url):
    """
    Acessa o site e extrai apenas o texto visível.
    Remove tags HTML.
    """

    class ExtratorDeTexto(html.parser.HTMLParser):
        """
        Classe auxiliar para extrair texto limpo do HTML.
        """

        def __init__(self):
            super().__init__()
            self.partes = []
            self.ignorar = False

        def handle_starttag(self, tag, attrs):
            # Ignora scripts e estilos
            if tag in ("script", "style"):
                self.ignorar = True

        def handle_endtag(self, tag):
            # Volta a capturar texto após script/style
            if tag in ("script", "style"):
                self.ignorar = False

        def handle_data(self, data):
            # Guarda apenas texto visível
            if not self.ignorar:
                texto = data.strip()

                if texto:
                    self.partes.append(texto)

        def obter_texto(self):
            # Junta todo o texto capturado
            return " ".join(self.partes)

    try:
        print("\nCarregando conteúdo do site...")

        # Simula navegador
        cabecalho = {
            "User-Agent": "Mozilla/5.0"
        }

        # Faz requisição ao site
        requisicao = urllib.request.Request(
            url,
            headers=cabecalho
        )

        resposta = urllib.request.urlopen(
            requisicao,
            timeout=15
        )

        # Lê HTML bruto
        html_bruto = resposta.read().decode(
            "utf-8",
            errors="ignore"
        )

        # Extrai texto
        extrator = ExtratorDeTexto()
        extrator.feed(html_bruto)

        texto = extrator.obter_texto()

        # Limita o tamanho para evitar excesso de tokens
        texto_limitado = texto[:8000]

        print("Conteúdo carregado com sucesso!\n")

        return texto_limitado

    except Exception as erro:
        print(f"Erro ao acessar o site: {erro}")
        return None


# -----------------------------------------------------------------------------
# PASSO 5 — CRIAR O ASSISTENTE (LLM)
# -----------------------------------------------------------------------------

def criar_assistente():
    """
    Cria o modelo de linguagem conectado ao Groq.
    """

    assistente = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.3
    )

    return assistente


# -----------------------------------------------------------------------------
# PASSO 6 — ENVIAR PERGUNTA PARA O ASSISTENTE
# -----------------------------------------------------------------------------

def fazer_pergunta(
    assistente,
    contexto_do_site,
    pergunta_do_usuario,
    nome_usuario
):
    """
    Envia a pergunta para o modelo junto com o contexto do site.
    """

    # Define comportamento do assistente
    mensagem_sistema = SystemMessage(
        content=(
            f"O nome do usuário é {nome_usuario}. "
            "Interaja de forma amigável, educada e personalizada. "
            "Use o nome do usuário quando fizer sentido.\n\n"

            "Você é um assistente especialista e deve responder "
            "usando APENAS as informações do conteúdo abaixo. "
            "Se não encontrar a resposta no conteúdo, diga isso honestamente. "
            "Responda sempre em português.\n\n"

            "=== CONTEÚDO DO SITE ===\n"
            f"{contexto_do_site}\n"
            "========================"
        )
    )

    # Pergunta do usuário
    mensagem_usuario = HumanMessage(
        content=pergunta_do_usuario
    )

    # Envia para o modelo
    resposta = assistente.invoke(
        [
            mensagem_sistema,
            mensagem_usuario
        ]
    )

    return resposta.content


# -----------------------------------------------------------------------------
# PASSO 7 — FUNÇÃO PRINCIPAL
# -----------------------------------------------------------------------------

def main():
    """
    Função principal do programa.
    """

    print("=" * 60)
    print("PRAGMATA AI")
    print("Modelo: llama 3.3")
    print("=" * 60)

    # ---------------------------------------------------
    # PEDIR NOME DO USUÁRIO
    # ---------------------------------------------------

    nome_usuario = input("\nDigite seu nome: ").strip()

    # Se usuário não digitar nome
    if not nome_usuario:
        nome_usuario = "Usuário"

    print(f"\nOlá, {nome_usuario}! Seja bem-vindo(a).")

    # ---------------------------------------------------
    # CARREGAR CONTEÚDO DO SITE
    # ---------------------------------------------------

    conteudo_do_site = extrair_texto_do_site(
        URL_DO_SITE
    )

    # Se não carregar, encerra
    if not conteudo_do_site:
        print("Não foi possível carregar o site.")
        return

    # ---------------------------------------------------
    # CRIAR ASSISTENTE
    # ---------------------------------------------------

    assistente = criar_assistente()

    print(f"Assistente pronto, {nome_usuario}!")
    print("Você pode fazer perguntas sobre o jogo Pragmata.")
    print("Digite 'sair' para encerrar.\n")

    # ---------------------------------------------------
    # LOOP PRINCIPAL
    # ---------------------------------------------------

    while True:

        pergunta = input(f"{nome_usuario}: ").strip()

        # Se vazio
        if not pergunta:
            print("Digite uma pergunta válida.\n")
            continue

        # Encerrar
        if pergunta.lower() == "sair":
            break

        try:
            print("\nConsultando assistente...\n")

            resposta = fazer_pergunta(
                assistente,
                conteudo_do_site,
                pergunta,
                nome_usuario
            )

            print(f"Assistente: {resposta}\n")
            print("-" * 60)

        except Exception as erro:
            print(f"Erro: {erro}")


    # ---------------------------------------------------
    # ENCERRAMENTO
    # ---------------------------------------------------

    print("\n" + "=" * 60)
    print(f"Obrigado por usar o Pragmata AI, {nome_usuario}!")
    print("Foi um prazer conversar com você.")
    print("Até a próxima!")
    print("=" * 60)


# -----------------------------------------------------------------------------
# PASSO 8 — EXECUTAR O PROGRAMA
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    main()

PRAGMATA AI
Modelo: llama 3.3

Digite seu nome: Do que se trata?

Olá, Do que se trata?! Seja bem-vindo(a).

Carregando conteúdo do site...
Conteúdo carregado com sucesso!

Assistente pronto, Do que se trata?!
Você pode fazer perguntas sobre o jogo Pragmata.
Digite 'sair' para encerrar.

Do que se trata?: Do que se trata?

Consultando assistente...

Assistente: Olá, Do que se trata! Estou aqui para ajudar. Você parece ter uma pergunta sobre o jogo Pragmata, mas não especificou o que gostaria de saber. Posso tentar ajudar com alguma informação geral sobre o jogo ou responder a uma pergunta específica que você possa ter.

Pragmata é um jogo de ação-aventura desenvolvido e publicado pela Capcom, lançado em 2026 para várias plataformas, incluindo PlayStation 5, Windows, Xbox Series X/S e Nintendo Switch 2. O jogo segue a história de Hugh e Diana, que trabalham juntos para lutar contra uma inteligência artificial hostil chamada IDUS em uma estação de pesquisa lunar.

Se você tiver alguma pe